# Grokking em Transformers: O Mecanismo de Atenção na Aritmética Modular

Este notebook demonstra o fenômeno de **grokking** em um **Transformer minimalista de 1 camada** treinado para realizar a adição modular:

$$a + b \pmod p$$

onde $p = 67$ é um número primo. 

A arquitetura do Transformer (base de modelos como GPT, Claude, e Gemini) utiliza o mecanismo de atenção causal para processar sequências. Mostramos que mesmo essa estrutura de rede neural altamente complexa exibe o comportamento de transição de fase dinâmico de grokking.

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Configurar sementes para reprodutibilidade
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando o dispositivo: {device}")

### 1. Arquitetura do Mini-Transformer

Implementamos um modelo decoder-only simples contendo:
- Camada de embedding de tokens e embeddings posicionais aprendidos.
- Camada de auto-atenção causal de 1 cabeça e dimensão de representação $64$.
- Uma camada MLP residual de dimensão oculta $128$.
- Layer Normalization nos sub-blocos.
- Camada linear final que prediz os $p$ resultados a partir da posição do token de igualdade `=`.

In [ ]:
class ModularTransformer(nn.Module):
    def __init__(self, p, embed_dim, num_heads, hidden_dim):
        super().__init__()
        self.num_tokens = p + 1 # p operandos + 1 token '=' representado pelo valor p
        self.embed = nn.Embedding(self.num_tokens, embed_dim)
        self.pos_embed = nn.Parameter(torch.randn(3, embed_dim))
        
        # Atenção linear
        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        
        # MLP residual
        self.fc1 = nn.Linear(embed_dim, hidden_dim)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden_dim, embed_dim)
        
        # Norms
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.unembed = nn.Linear(embed_dim, p, bias=False)
        
    def forward(self, x):
        # x shape: (B, 3) correspondente a [a, b, =]
        h = self.embed(x) + self.pos_embed.unsqueeze(0)
        
        # Auto-atenção com máscara causal
        h_norm = self.ln1(h)
        q = self.q_proj(h_norm)
        k = self.k_proj(h_norm)
        v = self.v_proj(h_norm)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / (q.shape[-1] ** 0.5)
        
        # Causal mask
        mask = torch.triu(torch.ones(3, 3, device=x.device), diagonal=1).bool()
        scores = scores.masked_fill(mask.unsqueeze(0), -1e9)
        
        weights = torch.softmax(scores, dim=-1)
        attn_out = torch.matmul(weights, v)
        attn_out = self.out_proj(attn_out)
        
        h = h + attn_out
        
        # MLP
        h_norm = self.ln2(h)
        mlp_out = self.fc2(self.act(self.fc1(h_norm)))
        h = h + mlp_out
        
        # Predição final no token '=' (última posição)
        logits = self.unembed(h[:, 2, :])
        return logits

### 2. Geração e Preparação dos Dados

O formato de cada exemplo é $[a, b, p]$ onde $p$ representa o caractere `=`. O target é $(a + b) \pmod p$.

In [ ]:
p = 67
train_fraction = 0.40 # Usamos 40% de treino para o Transformer

x_data = []
y_data = []
for a in range(p):
    for b in range(p):
        x_data.append([a, b, p]) # Formato [a, b, =]
        y_data.append((a + b) % p)

x_data = torch.tensor(x_data, dtype=torch.long)
y_data = torch.tensor(y_data, dtype=torch.long)

num_samples = len(x_data)
indices = np.random.permutation(num_samples)
split_idx = int(num_samples * train_fraction)

train_idx = indices[:split_idx]
val_idx = indices[split_idx:]

x_train, y_train = x_data[train_idx], y_data[train_idx]
x_val, y_val = x_data[val_idx], y_data[val_idx]

print(f"Dataset total: {num_samples} combinações")
print(f"Conjunto de treino: {len(x_train)}")
print(f"Conjunto de validação: {len(x_val)}")

### 3. Loop de Treinamento com Regularização Forte

Configuramos AdamW, taxa de aprendizado $10^{-3}$, e decaimento de peso $\lambda = 2.0$, ideal para acelerar a generalização no Transformer.

In [ ]:
model = ModularTransformer(p=p, embed_dim=64, num_heads=2, hidden_dim=128).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=2.0)
criterion = nn.CrossEntropyLoss()

epochs = 8000
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

x_train, y_train = x_train.to(device), y_train.to(device)
x_val, y_val = x_val.to(device), y_val.to(device)

for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    logits = model(x_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()
    
    model.eval()
    with torch.no_grad():
        train_acc = (logits.argmax(dim=-1) == y_train).float().mean().item()
        val_logits = model(x_val)
        val_loss = criterion(val_logits, y_val).item()
        val_acc = (val_logits.argmax(dim=-1) == y_val).float().mean().item()
        
    history['train_loss'].append(loss.item())
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    if (epoch + 1) % 500 == 0 or epoch == 0:
        print(f"Época {epoch+1:4d}/{epochs} | "
              f"Loss Treino: {loss.item():.4f} | Acc Treino: {train_acc*100:6.2f}% | "
              f"Loss Val: {val_loss:.4f} | Acc Val: {val_acc*100:6.2f}%")
        
        # Parada antecipada se generalizar
        if train_acc > 0.999 and val_acc > 0.99:
            if len(history['val_acc']) > 10 and all(a > 0.98 for a in history['val_acc'][-5:]):
                print(f"\nGrokking atingido precocemente na época {epoch+1}!")
                break

### 4. Resultados Gráficos

Visualização do comportamento do Transformer. Note que o Transformer grokka substancialmente mais rápido que o MLP sob decaimento de pesos de $2.0$.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6.5, 6.5), sharex=True)

# Perda
ax1.plot(history['train_loss'], label='Treino', color='blue', alpha=0.7)
ax1.plot(history['val_loss'], label='Validação', color='red', alpha=0.7)
ax1.set_yscale('log')
ax1.set_title('Perda vs Épocas (Transformer)')
ax1.set_ylabel('Perda Log')
ax1.legend()
ax1.grid(True, which='both', ls='--')

# Acurácia
ax2.plot(history['train_acc'], label='Treino', color='blue', alpha=0.7)
ax2.plot(history['val_acc'], label='Validação', color='red', alpha=0.7)
ax2.set_title('Acurácia vs Épocas (Transformer)')
ax2.set_xlabel('Épocas')
ax2.set_ylabel('Acurácia (%)')
ax2.legend()
ax2.grid(True, ls='--')

plt.suptitle('Grokking como Transição de Fase Dinâmica em Transformer', fontsize=12, y=0.99)
plt.tight_layout()
plt.show()